In [137]:
import re

import pandas as pd


In [138]:
df = pd.read_excel("data/Working file - Coforization -20.04.xlsx")


In [139]:
WANTED_COLUMNS = ["Seller COFOR2", "Manufacturer COFOR", "Manufacturer address", "Shipper COFOR2",
                  "Shipper COFOR Address", "Location ID2", "Location ID Address", "Quality contact", "Logistic contact",
                  "Contacted", "Info completed", "NOTE", "Owner", "Format check", "Status", "SOM double-check"]


In [140]:
LOCATION_COLUMNS = ["Manufacturer address", "Shipper COFOR Address", "Location ID Address"]


In [141]:
EMAIL_COLUMNS = ["Quality contact", "Logistic contact"]


In [142]:
CHAR_LENGTH_COLUMNS = ["Seller COFOR2", "Manufacturer COFOR", "Shipper COFOR2"]


In [143]:
CHAR_LENGTH_COLUMN_12 = ["Location ID2"]


In [144]:
EMAIL_REGEX = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'


In [145]:
CHAR_PATTERN_REGEX = r'^[a-zA-Z0-9]{6} {2}[a-zA-Z0-9]{2}$'


In [146]:
CHAR_LENGTH = 12


In [147]:
CONTACTED_ALLOWED_VALUES = ["yes", "no", "out of scope"]


In [148]:
def check_column_length(value) -> bool:
    if pd.isna(value) or not isinstance(value, str):
        return False
    return len(value.strip()) == CHAR_LENGTH


In [149]:
def check_column_against_regex(value, regex) -> bool:
    if pd.isna(value) or not isinstance(value, str):
        return False
    return re.match(regex, value.strip()) is not None


In [150]:
def normalize(df, wanted_columns) -> pd.DataFrame:
    for col in wanted_columns:
        df[col] = df[col].astype(str)
        df[col] = df[col].str.strip()
    return df


In [151]:
def build_comment_for_row(index) -> str:
    reasons = []

    failed_emails = [col for col in EMAIL_COLUMNS if fail_EMAIL_COLUMNS.at[index, col]]
    if failed_emails:
        reasons.append(f"Invalid email: {', '.join(failed_emails)}")

    failed_patterns = [col for col in CHAR_LENGTH_COLUMNS if fail_COLUMN_LENGTH.at[index, col]]
    if failed_patterns:
        reasons.append(f"Invalid COFOR pattern (6 chars + 2 spaces + 2 chars): {', '.join(failed_patterns)}")

    failed_len_12 = [col for col in CHAR_LENGTH_COLUMN_12 if fail_COLUMN_LENGTH_12.at[index, col]]
    if failed_len_12:
        reasons.append(f"Invalid length (must be {CHAR_LENGTH}): {', '.join(failed_len_12)}")

    if fail_CONTACTED.at[index]:
        reasons.append("Invalid Contacted value (allowed: yes, no, out of scope)")

    return " | ".join(reasons)


In [163]:
def check_column_against_allowed_values(value, allowed_values) -> bool:
    if pd.isna(value) or not isinstance(value, str):
        return False
    return str(value).strip().lower() in allowed_values


In [153]:
df["Check"] = pd.Series(dtype='boolean')
df["Comment"] = pd.Series(dtype='string')


In [154]:
df_normalized = normalize(df, WANTED_COLUMNS)


In [155]:
fail_EMAIL_COLUMNS = ~df_normalized.apply(
    lambda col: col.apply(
        lambda x: check_column_against_regex(x,EMAIL_REGEX) if col.name in EMAIL_COLUMNS else True
    )
)


In [156]:
fail_COLUMN_LENGTH = ~df_normalized.apply(
    lambda col: col.apply(
        lambda x: check_column_against_regex(x,CHAR_PATTERN_REGEX) if col.name in CHAR_LENGTH_COLUMNS else True
    )
)


In [157]:
fail_COLUMN_LENGTH_12 = ~df_normalized.apply(
    lambda col: col.apply(
        lambda x: check_column_length(x) if col.name in CHAR_LENGTH_COLUMN_12 else True
    )
)


In [164]:
fail_CONTACTED = ~df_normalized["Contacted"].apply(
    lambda x: check_column_against_allowed_values(
        x, CONTACTED_ALLOWED_VALUES
    )
)


In [171]:
# Count total failed checks per row (email + column-length).
df_normalized["Check"] = fail_EMAIL_COLUMNS.sum(axis=1).astype(int) + fail_COLUMN_LENGTH.sum(axis=1).astype(int) + fail_COLUMN_LENGTH_12.sum(axis=1).astype(int) + fail_CONTACTED.astype(int)


In [172]:
df_normalized["Comment"] = df.index.to_series().apply(build_comment_for_row).astype("string")


In [173]:
df_normalized["Check"].value_counts()


Check
0    15402
1     8743
2     6889
5     6818
3     4731
4      417
6      204
7        4
Name: count, dtype: int64


In [178]:
df_normalized[df_normalized["Check"] == 6].to_excel("data/failed_rows.xlsx", index=False)
